In [1]:
# %pip install --upgrade transformers --break-system-packages

In [2]:
# %pip install auto-round --break-system-packages

In [3]:
# %pip install compressed-tensors --break-system-packages

In [4]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [5]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [6]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from safetensors import safe_open
from transformers import AutoModelForImageTextToText, AutoProcessor


In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [8]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [9]:
notebook_login()

In [10]:
MODEL_ID = "Qwen/Qwen3.5-4B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [11]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 14 files: 100%|██████████████████████| 14/14 [00:00<00:00, 410.32it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [12]:
for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00001-of-00002.safetensors
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight

Checking model.safetensors-00002-of-00002.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [13]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [14]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [ ]:

# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1000,      # Optimization iterations per block; balances calibration time and accuracy
    group_size=16,   # Number of weights sharing one scale. Use 16 when targeting NVFP4.
    sym=True,        # Symmetric quantization; enables faster compute and broader engine compatibility
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                         # Processor for multimodal models (vision/audio), if present
    scheme="NVFP4",                             # NVIDIA FP4 quantization format for reduced memory usage and accelerated inference on supported hardware
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                               # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,                               # Calibration batch size; balances processing speed and GPU memory usage
    gradient_accumulate_steps=4,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
    seqlen=2048,                                # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,                    # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision
    enable_torch_compile=False,                 # Disables torch.compile to avoid the NVFP4/static warning
    ignore_layers="linear_attn,mtp,visual",     # Excludes linear-attention, MTP, and visual modules from quantization
    layer_config={
        "mtp": {"data_type": "bfloat16"},       # Keeps the MTP module in BF16 precision
        "mtp.fc": {"data_type": "bfloat16"},    # Keeps the MTP fully connected layer in BF16 precision
    },                                          # Preserves BF16 precision for the specified MTP layers
)


2026-10-15 17:29:02 WARNING autoround.py L489: Passing 'gradient_accumulate_steps' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [16]:
ar.quantize_and_save(OUTPUT_BASE_DIR, format="llm_compressor", inplace=True)

2026-10-15 17:29:03 INFO layer_config_resolver.py L402: Ignored layers: linear_attn, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_a, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_b, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_qkv, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_z, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.out_proj, model.visual.blocks.[0-23].attn.proj, model.visual.blocks.[0-23].attn.qkv, model.visual.blocks.[0-23].mlp.linear_fc1, model.visual.blocks.[0-23].mlp.linear_fc2, model.visual.merger.linear_fc1, model.visual.merger.linear_fc2, mtp, visual
2026-10-15 17:29:03 INFO layer_config_resolver.py L402: Ignored layers: linear_attn, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_a, model.language_m

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): 

In [17]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [18]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [19]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-nvfp-w4g16"),
        f"{base_name}-W4A16-AutoRound-NVFP4",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-nvfp-w4g16 to Vishva007/Qwen3.5-4B-W4A16-AutoRound-NVFP4...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-4B-W4A16-AutoRound-NVFP4
